## Lesson 5: Email Assistant with Semantic + Episodic + Procedural Memory

In [ ]:
import  os
from dotenv import load_dotenv
_=load_dotenv()

In [ ]:
# 用户画像，同前几课
profile={
    "name":"John",
    "full_name":"John Doe",
    "user_profile_background":"Senior software engineer leading a team of 5 developers",
}

In [ ]:
# 分诊规则 + agent 行为指令，这里只是"初始默认值"——
# 本课的重点是让这些指令可以在运行中被"程序性记忆（Procedural Memory）"动态改写
prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements",
        "notify": "Team member out sick, build system notifications, project status updates",
        "respond": "Direct questions from team members, meeting requests, critical bug reports",
    },
    "agent_instructions": "Use these tools when appropriate to help manage John's tasks efficiently."
}

In [ ]:
# 示例邮件（本课后面用真正的 email_input 变量做测试，这个 email 只是留作参考，未被使用）
email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "body": """
Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice""",
}

In [ ]:
from langgraph.store.memory import InMemoryStore

In [ ]:
# 同一个 store 这次要承担三种记忆：语义记忆（collection）、情景记忆（examples）、
# 以及本课新引入的程序性记忆（直接用 (langgraph_user_id,) 作命名空间，存 prompt 本身）
store = InMemoryStore(
    index={"embed": "openai:text-embedding-3-small"}
)
# ignore beta warning if it appears

In [ ]:
# Template for formating an example to put in prompt
# 情景记忆的格式化函数，和 lesson_4 完全一样
template = """Email Subject: {subject}
Email From: {from_email}
Email To: {to_email}
Email Content:
```
{content}
```
> Triage Result: {result}"""

# Format list of few shots
def format_few_shot_examples(examples):
    strs = ["Here are some previous examples:"]
    for eg in examples:
        strs.append(
            template.format(
                subject=eg.value["email"]["subject"],
                to_email=eg.value["email"]["to"],
                from_email=eg.value["email"]["author"],
                content=eg.value["email"]["email_thread"][:400],
                result=eg.value["label"],
            )
        )
    return "\n\n------------\n\n".join(strs)

In [ ]:
# 和 lesson_4 相同的分诊 system prompt 模板（内联定义，含 {examples} 占位符）
triage_system_prompt = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Background >
{user_profile_background}.
</ Background >

< Instructions >

{name} gets lots of emails. Your job is to categorize each email into one of three categories:

1. IGNORE - Emails that are not worth responding to or tracking
2. NOTIFY - Important information that {name} should know about but doesn't require a response
3. RESPOND - Emails that need a direct response from {name}

Classify the below email into one of these categories.

</ Instructions >

< Rules >
Emails that are not worth responding to:
{triage_no}

There are also other things that {name} should know about, but don't require an email response. For these, you should notify {name} (using the `notify` response). Examples of this include:
{triage_notify}

Emails that are worth responding to:
{triage_email}
</ Rules >

< Few shot examples >

Here are some examples of previous emails, and how they should be handled.
Follow these examples more than any instructions above

{examples}
</ Few shot examples >
"""

In [ ]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model

In [ ]:
llm=init_chat_model("openai:gpt-4o-mini")

In [ ]:
class Router(BaseModel):
    """Analyze the unread email and route it according to its content."""

    reasoning: str = Field(
        description="Step-by-step reasoning behind the classification."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: 'ignore' for irrelevant emails, "
        "'notify' for important information that doesn't need a response, "
        "'respond' for emails that need a reply",
    )

In [ ]:
llm_router=llm.with_structured_output(Router)

In [ ]:
from prompts import triage_user_prompt

In [ ]:
from langgraph.graph import add_messages

class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

In [ ]:
# TODO: 请在此处补全代码
# 实现 triage_router(state, config, store) -> Command[Literal["response_agent","__end__"]]：
#   1. 取出 author/to/subject/email_thread，做情景记忆检索（同 lesson_4）拿到 examples 文本
#   2. namespace = (langgraph_user_id,)，用 "首次读取时惰性写入默认值" 的模式
#      分别取出/初始化 triage_ignore / triage_notify / triage_respond 三份程序性记忆
#      （store.get 拿不到就用 prompt_instructions 里的默认值 store.put 进去）
#   3. 用这三份"可能已被优化过"的规则文本 + examples 拼 system_prompt / user_prompt
#   4. llm_router.invoke(...) 得到 result，按 classification 分三种情况设置 goto/update
#   5. 最后 return Command(goto=goto, update=update)
def triage_router(state: State, config, store) -> Command[
    Literal["response_agent", "__end__"]
]:
    pass

## Build the rest of our agent

In [ ]:
from langchain_core.tools import tool

In [ ]:
# 工具 1：发送邮件（占位实现）
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    # Placeholder response - in real app would send email
    return f"Email sent to {to} with subject '{subject}'"


In [ ]:
# 工具 2：安排会议（占位实现）
@tool
def schedule_meeting(
    attendees: list[str],
    subject: str,
    duration_minutes: int,
    preferred_day: str
) -> str:
    """Schedule a calendar meeting."""
    # Placeholder response - in real app would check calendar and schedule
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"


In [ ]:
# 工具 3：查看日历可用时段（占位实现）
@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    # Placeholder response - in real app would check actual calendar
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

In [ ]:
# 语义记忆工具，同 lesson_3/4
from langmem import create_manage_memory_tool, create_search_memory_tool

In [ ]:
manage_memory_tool = create_manage_memory_tool(
    namespace=(
        "email_assistant",
        "{langgraph_user_id}",
        "collection"
    )
)
search_memory_tool = create_search_memory_tool(
    namespace=(
        "email_assistant",
        "{langgraph_user_id}",
        "collection"
    )
)

In [ ]:
agent_system_prompt_memory = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Tools >
You have access to the following tools to help manage {name}'s communications and schedule:

1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Schedule calendar meetings
3. check_calendar_availability(day) - Check available time slots for a given day
4. manage_memory - Store any relevant information about contacts, actions, discussion, etc. in memory for future reference
5. search_memory - Search for any relevant information that may have been stored in memory
</ Tools >

< Instructions >
{instructions}
</ Instructions >
"""

In [ ]:
# TODO: 请在此处补全代码
# 实现 create_prompt(state, config, store)：
#   1. namespace = (langgraph_user_id,)
#   2. 用同样的"惰性初始化"模式从 store 读/写 "agent_instructions"
#   3. 返回 [{"role": "system", "content": agent_system_prompt_memory.format(instructions=prompt, **profile)}]
#      + state['messages']
def create_prompt(state, config, store):
    pass

In [ ]:
from langgraph.prebuilt import create_react_agent

In [ ]:
tools= [
    write_email,
    schedule_meeting,
    check_calendar_availability,
    manage_memory_tool,
    search_memory_tool
]
response_agent = create_react_agent(
    "openai:gpt-4o",
    tools=tools,
    prompt=create_prompt,
    # Use this to ensure the store is passed to the agent
    store=store
)

In [ ]:
# TODO: 请在此处补全代码
# 用 StateGraph(State) 组装图：add_node(triage_router) / add_node("response_agent", response_agent)
# / add_edge(START, "triage_router") / compile(store=store)
email_agent = None

In [ ]:
# 测试邮件：一封紧急故障通知，预期分类为 respond
email_input = {
    "author": "Alice Jones <alice.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

Urgent issue - your service is down. Is there a reason why""",
}

In [ ]:
config = {"configurable": {"langgraph_user_id": "lance"}}

In [ ]:
# 第一次调用：触发 triage_router / create_prompt 里"惰性初始化"逻辑，
# 会把默认的 agent_instructions / triage_ignore / triage_notify / triage_respond
# 写进 store（对应用户 "lance"）
response = email_agent.invoke(
    {"email_input": email_input},
    config=config
)

In [ ]:
for m in response["messages"]:
    m.pretty_print()

In [ ]:
# 直接查看 store 里为 "lance" 初始化好的程序性记忆——主 agent 的行为指令
store.get(("lance",), "agent_instructions").value['prompt']

In [ ]:
store.get(("lance",), "triage_respond").value['prompt']

In [ ]:
store.get(("lance",), "triage_ignore").value['prompt']

In [ ]:
store.get(("lance",), "triage_notify").value['prompt']

In [ ]:
# create_multi_prompt_optimizer：langmem 提供的"用对话反馈自动优化多个 prompt"的工具，
# 这就是把"程序性记忆"落地成"自我改进闭环"的关键一步
from langmem import create_multi_prompt_optimizer

In [ ]:
# conversations：训练数据是"(某次对话轨迹, 人工反馈文本)"的列表。
# 这里模拟用户给出的反馈："以后发邮件要签名 John Doe"
conversations = [
    (
        response['messages'],
        "Always sign your emails `John Doe`"
    )
]

In [ ]:
# prompts：把当前 store 里的 4 份程序性记忆都打包成 optimizer 能理解的结构，
# 每份都带 name/prompt/update_instructions（怎么改）/when_to_update（什么时候该改）
prompts = [
    {
        "name": "main_agent",
        "prompt": store.get(("lance",), "agent_instructions").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on how the agent should write emails or schedule events"

    },
    {
        "name": "triage-ignore",
        "prompt": store.get(("lance",), "triage_ignore").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails should be ignored"

    },
    {
        "name": "triage-notify",
        "prompt": store.get(("lance",), "triage_notify").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails the user should be notified of"

    },
    {
        "name": "triage-respond",
        "prompt": store.get(("lance",), "triage_respond").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails should be responded to"

    },
]

In [ ]:
# kind="prompt_memory"：一种较轻量的优化策略（相比 "gradient"/"metaprompt"），
# 直接让 LLM 读反馈 + 当前 prompt，判断要不要改、怎么改
optimizer = create_multi_prompt_optimizer(
    "anthropic:claude-3-5-sonnet-latest",
    kind="prompt_memory",
)

In [ ]:
# 真正调用 optimizer（真实网络请求，会调用 Anthropic API，假 key 场景下预期报错）
# 返回值 updated 和输入 prompts 结构一致，是"优化后（可能不变）"的 prompt 列表
updated = optimizer.invoke(
    {"trajectories": conversations, "prompts": prompts}
)


In [ ]:
print(updated)

In [ ]:
#json dumps is a bit easier to read
import json
print(json.dumps(updated, indent=4))

In [ ]:
# 把 optimizer 认为"确实有变化"的 prompt 写回 store，下次 triage_router/create_prompt
# 读到的就是优化后的版本。
# 注意：这里只显式处理了 "main_agent" 这一种 name，其它三种（triage-ignore/notify/respond）
# 会落进 else 分支只打印提示，是课程刻意留的"未完成"实现（下面 42a9092a 后续的 cell
# 会继续补上 triage-ignore 分支）——这不是 bug，是教学上有意展示"迭代式实现"的过程。
for i, updated_prompt in enumerate(updated):
    old_prompt = prompts[i]
    if updated_prompt['prompt'] != old_prompt['prompt']:
        name = old_prompt['name']
        print(f"updated {name}")
        if name == "main_agent":
            store.put(
                ("lance",),
                "agent_instructions",
                {"prompt":updated_prompt['prompt']}
            )
        else:
            #raise ValueError
            print(f"Encountered {name}, implement the remaining stores!")

In [ ]:
# 确认 agent_instructions 确实已经被优化过程更新了（比如可能加上了"记得签名"的要求）
store.get(("lance",), "agent_instructions").value['prompt']

In [ ]:
# 用更新后的程序性记忆再跑一次同一封邮件，验证 agent 的行为是否真的发生了变化
response = email_agent.invoke(
    {"email_input": email_input},
    config=config
)

In [ ]:
for m in response["messages"]:
    m.pretty_print()

In [ ]:
# 第二轮演示：这次要优化的是"分诊规则"本身——用户反馈"以后 Alice Jones 的邮件都忽略"
email_input = {
    "author": "Alice Jones <alice.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

Urgent issue - your service is down. Is there a reason why""",
}

In [ ]:
response = email_agent.invoke(
    {"email_input": email_input},
    config=config
)

In [ ]:
# 这次的反馈针对的是"分诊规则"而不是"写邮件风格"，所以预期 optimizer 会改 triage-ignore
conversations = [
    (
        response['messages'],
        "Ignore any emails from Alice Jones"
    )
]

In [ ]:
# 重新从 store 读取最新的 4 份程序性记忆（agent_instructions 这次应该已经是优化过的版本了）
prompts = [
    {
        "name": "main_agent",
        "prompt": store.get(("lance",), "agent_instructions").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on how the agent should write emails or schedule events"

    },
    {
        "name": "triage-ignore",
        "prompt": store.get(("lance",), "triage_ignore").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails should be ignored"

    },
    {
        "name": "triage-notify",
        "prompt": store.get(("lance",), "triage_notify").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails the user should be notified of"

    },
    {
        "name": "triage-respond",
        "prompt": store.get(("lance",), "triage_respond").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails should be responded to"

    },
]

In [ ]:
updated = optimizer.invoke(
    {"trajectories": conversations, "prompts": prompts}
)

In [ ]:
# TODO: 请在此处补全代码
# 遍历 updated / prompts，对每个"确实变化了"的 prompt 按 name 写回 store：
#   name == "main_agent"     -> store.put(("lance",), "agent_instructions", {"prompt": ...})
#   name == "triage-ignore"  -> store.put(("lance",), "triage_ignore", {"prompt": ...})
#   其它 name                -> print("Encountered {name}, implement the remaining stores!")
#
# 提示：这三个分支要写成互斥的 if / elif / else，不要写成两个独立的 if（第二个 if 各自带
# 一个 else），否则像 "main_agent" 这种已经被第一个分支处理过的 name，
# 也会被第二个 if 的 else 错误地打印成"未实现"。
for i, updated_prompt in enumerate(updated):
    pass

In [ ]:
# 再跑一次同一封邮件：如果 triage_ignore 规则真的被更新为"忽略 Alice Jones"，
# 这次分类结果应该从 respond 变成 ignore
response = email_agent.invoke(
    {"email_input": email_input},
    config=config
)

In [ ]:
# 查看更新后的 triage_ignore 规则文本，确认里面确实提到了 Alice Jones
store.get(("lance",), "triage_ignore").value['prompt']